> **What this notebook is:** a survivorship-bias / PIT-correction quantification on the headline models (Lasso-6, Ridge-6, LightGBM rank-9).
>
> It runs two arms to show how much apparent IC comes from looking only at today's S&P 500 survivors vs the historical PIT-correct universe:
> 1. **PIT-on (honest = headline)** — per-date S&P 500 membership.
> 2. **Survivor-snapshot (biased)** — the CURRENT roster, used across all history.
>
> Both arms draw from the same universe (`sp500_pit`) and differ ONLY in which cross-section is used on each date.


## Prerequisites
No prediction store runs are required. This notebook fits the models directly in memory for the 2025+ test slice to dynamically ablate the universes, identical to `scripts/survivorship_ablation_headline.py` (excluding the all-available baseline).


In [8]:
import warnings
from datetime import date
from pathlib import Path

import polars as pl
import yaml
import pandas as pd

from price_model.data.loaders import load_panel
from price_model.data.membership import filter_panel_to_pit, members_on_date
from price_model.eval.turnover import compute_turnover_and_costs
from price_model.features.pipeline import build_feature_matrix, drop_warmup_rows
from price_model.models import build_model
from price_model.models.base import ModelConfig
from price_model.pipeline.walk_forward import join_with_realized, run_walk_forward

warnings.filterwarnings("ignore")

FEATS6 = ["momentum_12_1", "momentum_756", "return_1d",
          "vol_ewm_20", "distance_52w_high", "log_dollar_volume"]
TRAIN_START, FIRST_REFIT, OOS_START = date(2022, 10, 10), date(2025, 1, 2), date(2025, 1, 1)


def find_repo_root(start: Path | None = None) -> Path:
    base = (start or Path.cwd()).resolve()
    for candidate in [base, *base.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "config").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


def lgbm_spec() -> tuple:
    repo_root = find_repo_root()
    cfg_path = repo_root / "config/experiments/lightgbm_rank9_h21_hp_pre20241231.yaml"
    if not cfg_path.exists():
        raise FileNotFoundError(f"Config file not found: {cfg_path}")
    with cfg_path.open("r", encoding="utf-8") as fh:
        cfg = yaml.safe_load(fh)
    params = next(m["params"] for m in cfg["models"] if m["class"] == "LightGBMModel")
    return ("LightGBM rank-9 (tuned)", list(cfg["features"]), cfg.get("normalize_kind", "rank"),
            "LightGBMModel", params)


def run_cell(spec: tuple, raw: pl.DataFrame) -> dict:
    name, feats, norm, cls, params = spec
    matrix = build_feature_matrix(raw, feats, norm, 21).pipe(drop_warmup_rows, feats).sort(["ticker", "date"])
    target = matrix.select("date", "ticker", "y")
    model = build_model(cls, ModelConfig(model_id="surv", feature_cols=tuple(feats), params=params))
    preds = run_walk_forward(
        matrix, model=model, feature_cols=feats, target_col="y",
        experiment_id="surv_ablation", horizon_days=21,
        refit_freq_days=9999, embargo_days=33, min_train_days=504,
        train_start=TRAIN_START, first_refit=FIRST_REFIT,
    )
    joined = join_with_realized(preds, target).filter(pl.col("date") >= OOS_START)
    s = compute_turnover_and_costs(
        joined.select("date", "ticker", "prediction", "realized"),
        cost_bps=(3, 10, 20), horizon_days=21,
    )
    return {"tickers": joined["ticker"].n_unique(), "ic": s.gross_ic, "t": s.gross_ic_t_stat,
            "sharpe": s.gross_long_short_sharpe, "turn": s.annual_turnover,
            "net20": s.after_cost_sharpe_by_bp[20]}

In [9]:
print("Loading sp500_pit price panel (no membership filter)...")
raw = load_panel(universe="sp500_pit", start="2017-01-01", pit_filter=False)
last = raw["date"].max()
current = members_on_date(last) & set(raw["ticker"].unique().to_list())

arms = {
    "PIT-on (honest)": filter_panel_to_pit(raw),
    "survivor-snapshot": raw.filter(pl.col("ticker").is_in(list(current))),
}

models = [
    ("Lasso-6", FEATS6, "rank", "LassoCrossSectional", {"cv": 3}),
    ("Ridge-6", FEATS6, "rank", "RidgeCrossSectional", {"cv": 3}),
    lgbm_spec(),
]


Loading sp500_pit price panel (no membership filter)...


$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-26) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-26) (Yahoo error = "No data found, symbol may be delisted")
$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-26) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-26) (Yahoo error = "No data found, symbol may be delisted")
$HOLX: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-26) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HOLX']: possibly delisted; no price data found  (1d 2026-04-07 -> 2026-06-26) (Yahoo error = "No data found, symbol may be delisted")
yfinance returned no data for HOLX after 3 attempts
$SEE: possibly delisted; no price data found  (1d 2026-04-09 -> 2026-06-26) (Yaho

In [10]:
rows = {}
for spec in models:
    mname = spec[0]
    for aname, araw in arms.items():
        print(f"Running {mname} × {aname}...")
        rows[(mname, aname)] = run_cell(spec, araw)


Running Lasso-6 × PIT-on (honest)...
Running Lasso-6 × survivor-snapshot...
Running Ridge-6 × PIT-on (honest)...
Running Ridge-6 × survivor-snapshot...
Running LightGBM rank-9 (tuned) × PIT-on (honest)...
Running LightGBM rank-9 (tuned) × survivor-snapshot...


In [11]:
results_df = []
for spec in models:
    mname = spec[0]
    for aname in arms:
        r = rows[(mname, aname)]
        results_df.append({
            "Model": mname,
            "Arm": aname,
            "Tickers": r["tickers"],
            "OOS IC": round(r["ic"], 4),
            "t": round(r["t"], 2),
            "Gross Sharpe": round(r["sharpe"], 2),
            "Ann. Turn": round(r["turn"], 0),
            "Net@20": round(r["net20"], 2)
        })

df = pd.DataFrame(results_df)
display(df)


,Model,Arm,Tickers,OOS IC,t,Gross Sharpe,Ann. Turn,Net@20
0,Lasso-6,PIT-on (honest),469,0.0872,10.09,1.96,10.0,1.70
1,Lasso-6,survivor-snapshot,497,0.0869,7.84,1.40,10.0,1.21
2,Ridge-6,PIT-on (honest),469,0.0871,10.09,1.96,10.0,1.70
3,Ridge-6,survivor-snapshot,497,0.0868,7.84,1.40,10.0,1.21
4,LightGBM rank-9 (tuned),PIT-on (honest),469,0.0689,6.04,1.37,11.0,1.09
5,LightGBM rank-9 (tuned),survivor-snapshot,497,0.0761,6.58,1.25,12.0,1.00


In [12]:
print("Survivorship inflation (survivor-snapshot vs PIT-on):")
for spec in models:
    mname = spec[0]
    h, s = rows[(mname, "PIT-on (honest)")], rows[(mname, "survivor-snapshot")]
    inflation_ic = s['ic'] - h['ic']
    inflation_pct = (inflation_ic / h['ic']) * 100 if h['ic'] else 0
    inflation_sharpe = s['sharpe'] - h['sharpe']
    print(f"{mname:24s} IC {inflation_ic:+.4f} ({inflation_pct:+.0f}%)   gross Sharpe {inflation_sharpe:+.2f}")


Survivorship inflation (survivor-snapshot vs PIT-on):
Lasso-6                  IC -0.0002 (-0%)   gross Sharpe -0.55
Ridge-6                  IC -0.0003 (-0%)   gross Sharpe -0.55
LightGBM rank-9 (tuned)  IC +0.0072 (+10%)   gross Sharpe -0.12
